# Scraping statistics

All numbers and figures on this page are computed when the book is built, from the files the crawl wrote
to `logs/` (`benchmark.json`, `papers.csv`). The definitions follow the lab book:

- **scraped successfully**: every version of the paper was downloaded and processed, including versions
  that only exist as PDF (they keep an empty record, as the lab book asks);
- **success rate**: over the assigned IDs, and over the papers that actually exist on arXiv
  (part of every assigned range is unused or withdrawn IDs);
- **size before / after**: unpacked size of all files of all versions of a paper, then the size of the
  kept `.tex` and `.bib` files; averaged over papers with at least one LaTeX version;
- **references per paper** and **reference success rate**: from each paper's final `references.json`
  (Semantic Scholar), over the papers on arXiv.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import report_data as data

data.check()
bench, papers = data.benchmark(), data.papers()
on_arxiv = papers[papers["versions"] > 0]
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML

PLOTLY_CONFIG = {"displaylogo": False, "responsive": True, "toImageButtonOptions": {"format": "png", "scale": 2}}


def show(fig, height=360):
    """An interactive figure in the built book: hover for the exact numbers, drag to zoom, double-click to reset,
    click a legend entry to hide it. plotly.js is loaded from its CDN."""
    fig.update_layout(template="plotly_white", height=height, margin=dict(l=10, r=10, t=60, b=10),
                      font=dict(size=12), hoverlabel=dict(font_size=12), title_x=0.0)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config=PLOTLY_CONFIG)))

COLOR, BEFORE, AFTER = "#3b7a8f", "#c96f53", "#1f4e5c"


def pct(value):
    return "–" if value is None else f"{value:.2f}%"

## Summary

In [ ]:
rows = [
    ("Assigned IDs", f"{bench['ids_assigned']:,}"),
    ("Papers that exist on arXiv", f"{bench['papers_on_arxiv']:,}"),
    ("Papers scraped successfully", f"{bench['papers_scraped']:,}"),
    ("Success rate (assigned IDs)", pct(bench["success_rate_of_assigned_pct"])),
    ("Success rate (papers on arXiv)", pct(bench["success_rate_of_papers_on_arxiv_pct"])),
    ("Source versions processed", f"{bench['versions_processed']:,} / {bench['versions_total']:,}"),
    ("Papers with LaTeX source", f"{bench['papers_with_latex']:,}"),
    ("Average paper size before figure removal", f"{bench['avg_paper_bytes_before']:,.0f} B "
                                                  f"({bench['avg_paper_bytes_before'] / 1024:,.1f} KB)"),
    ("Average paper size after (.tex/.bib only)", f"{bench['avg_paper_bytes_after']:,.0f} B "
                                                   f"({bench['avg_paper_bytes_after'] / 1024:,.1f} KB)"),
    ("Space saved", pct(bench["space_saved_pct"])),
    ("Average references per paper", f"{bench['avg_references_per_paper']:.2f}"),
    ("Average arXiv references per paper", f"{bench['avg_arxiv_references_per_paper']:.2f}"),
    ("Reference metadata success rate", pct(bench["reference_success_rate_pct"])),
]
display(pd.DataFrame(rows, columns=["Statistic", "Value"]).style.hide(axis="index"))

## Outcome of every assigned ID

In [ ]:
labels = {"latex": "LaTeX for every version", "mixed": "LaTeX for some versions",
          "pdf_only": "PDF only (empty record)", "incomplete": "a version still missing",
          "not_on_arxiv": "ID not on arXiv"}
counts = papers["outcome"].value_counts().reindex(labels).fillna(0).astype(int)
fig = go.Figure(go.Bar(
    x=counts.values, y=[labels[k] for k in counts.index], orientation="h", marker_color=COLOR,
    text=[f"{v:,} ({v / len(papers):.1%})" for v in counts.values], textposition="outside", cliponaxis=False,
    customdata=counts.values / len(papers), hovertemplate="%{y}: %{x:,} IDs (%{customdata:.2%})<extra></extra>"))
fig.update_layout(title="What each assigned ID turned out to be", xaxis_title="Assigned IDs",
                  xaxis_range=[0, counts.max() * 1.3], yaxis_autorange="reversed")
show(fig, height=320)
display(Markdown(
    f"Assigned IDs: **{len(papers):,}**, of which **{counts['not_on_arxiv']:,}** do not exist on arXiv. "
    f"Papers on arXiv: **{len(on_arxiv):,}**; with LaTeX source: **{counts['latex'] + counts['mixed']:,}**; "
    f"PDF-only: **{counts['pdf_only']:,}** (they keep an empty version folder, so the paper is not dropped); "
    f"still missing a version: **{counts['incomplete']:,}** (listed in `state/download/errors.jsonl`)."))

## Versions per paper

In [ ]:
versions = on_arxiv["versions"].astype(int).value_counts().sort_index()
fig = go.Figure(go.Bar(
    x=[f"v{v}" for v in versions.index], y=versions.values, marker_color=COLOR,
    text=[f"{v:,}" for v in versions.values], textposition="outside", cliponaxis=False,
    customdata=versions.values / versions.sum(), hovertemplate="%{x}: %{y:,} papers (%{customdata:.2%})<extra></extra>"))
fig.update_layout(title="Versions per paper", xaxis_title="Number of versions on arXiv",
                  yaxis_title="Papers (log scale)", yaxis_type="log")
show(fig)
single = versions.get(1, 0) / versions.sum()
display(Markdown(
    f"**{single:.1%}** of the papers have a single version; the average is "
    f"**{on_arxiv['versions'].mean():.2f}** versions per paper and the maximum **{int(on_arxiv['versions'].max())}**. "
    f"Every version is downloaded separately (`/src/{{id}}v{{N}}`), so the number of source requests is "
    f"**{int(on_arxiv['versions'].sum()):,}**."))

## Paper size before and after figure removal

In [ ]:
sized = on_arxiv[(on_arxiv["versions_ok"] > 0) & (on_arxiv["size_before"] > 0)]
low = max(1, min(sized["size_after"].min(), sized["size_before"].min()))
bins = np.geomspace(low, sized["size_before"].max() * 1.01, 40)   # log-spaced over the data
fig = go.Figure()
for column, colour, name, fill in (("size_before", BEFORE, "before (all files)", "tozeroy"),
                                   ("size_after", AFTER, "after (.tex/.bib)", None)):
    counts, edges = np.histogram(sized[column], bins=bins)
    fig.add_trace(go.Scatter(
        x=edges[:-1], y=counts, mode="lines", line_shape="hv", line=dict(color=colour, width=2), fill=fill,
        name=name, customdata=np.c_[edges[:-1] / 1024, edges[1:] / 1024],
        hovertemplate=name + ": %{y:,} papers between %{customdata[0]:,.0f} and %{customdata[1]:,.0f} KB"
                             "<extra></extra>"))
fig.update_layout(title="Size per paper before and after removing figures", xaxis_type="log",
                  xaxis_title="Size per paper, all versions (bytes, log scale)", yaxis_title="Papers",
                  legend=dict(orientation="h", y=-0.2))
show(fig, height=400)
display(Markdown(
    f"Over **{len(sized):,}** papers with LaTeX, the median size drops from "
    f"**{sized['size_before'].median() / 1024:,.0f} KB** to **{sized['size_after'].median() / 1024:,.0f} KB**; "
    f"on average **{bench['space_saved_pct']:.1f}%** of every paper is figures and other non-source files "
    f"(images, PDFs, style files), which never reach Google Drive."))

## References per paper

In [ ]:
with_refs = on_arxiv.dropna(subset=["n_refs"])
top = max(10, int(with_refs["n_refs"].quantile(0.99)))
fig = go.Figure()
for column, colour, name, opacity in (("n_refs", COLOR, "all references", 0.65),
                                      ("n_arxiv_refs", BEFORE, "references with an arXiv ID", 0.65)):
    fig.add_trace(go.Histogram(
        x=with_refs[column].clip(upper=top), xbins=dict(start=0, end=top + 5, size=5), name=name,
        marker_color=colour, opacity=opacity,
        hovertemplate=name + ": %{y:,} papers with %{x} references<extra></extra>"))
fig.update_layout(title="References per paper (Semantic Scholar)", barmode="overlay",
                  xaxis_title=f"References per paper (last bin: {top} or more)", yaxis_title="Papers",
                  legend=dict(orientation="h", y=-0.2))
show(fig, height=400)
share = with_refs["n_arxiv_refs"].sum() / with_refs["n_refs"].sum() if with_refs["n_refs"].sum() else 0
display(Markdown(
    f"Semantic Scholar returned a reference list for **{bench['references_ok']:,}** papers "
    f"({pct(bench['reference_success_rate_pct'])} of the papers on arXiv); **{bench['references_not_found']:,}** "
    f"are not indexed by Semantic Scholar (recent papers can take time to be indexed). On average a paper "
    f"cites **{bench['avg_references_per_paper']:.1f}** works, **{share:.1%}** of them with an arXiv ID; only "
    f"those become rows of the `references` table, but every reference is kept in `references.json`."))